# Loan Approval - exploratory prototype  *(RESEARCH CODE)*

**Group 84 | AIMLCZG546 SEML | Assignment II**

This notebook is the *before* artefact for Objective 1.2. It is genuine
exploratory code: it was written to answer "is there signal here?" as fast as
possible, and it succeeds at that. It is also unfit to serve traffic, and the
report explains exactly why, defect by defect.

**Do not fix this notebook.** Its value is as evidence. The engineered
counterpart is `src/loan_risk/features/engineering.py`.


## 1. Load the data

Path is whatever was on the analyst's laptop that afternoon.

In [ ]:
import pandas as pd, numpy as np
import os, sys, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

df = pd.read_csv("C:/Users/analyst/Desktop/loan_data_processed.csv")   # hardcoded local path
print(df.shape)
print( df.head() )

## 2. Poke at it

In [ ]:
tmp = df.describe()
X = df.drop('LoanApproved',axis=1)
y = df['LoanApproved']

## 3. Feature ideas

Try a few, keep whatever looks good.

In [ ]:
df['LoanToIncomeRatio'] = df['LoanAmount']/df['AnnualIncome']       # NOTE: blows up when income is 0
df['SavingsToLoanRatio'] = df['SavingsAccountBalance']/df['LoanAmount']
df['ratio2'] = df['MonthlyDebtPayments']*12/df['AnnualIncome']
df['x'] = df['CreditScore']/850
FEATURES=['Age','AnnualIncome','CreditScore','LoanAmount','LoanToIncomeRatio','SavingsToLoanRatio','ratio2','x','DebtToIncomeRatio','CreditCardUtilizationRate','BankruptcyHistory','PreviousLoanDefaults']

## 4. Fit something

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df[FEATURES],y,test_size=0.2)      # no random_state -> not reproducible
m = RandomForestClassifier()
m.fit(X_train,y_train)
p=m.predict(X_test)
print("acc",accuracy_score(y_test,p))

In [ ]:
def get_ratio(a,b) :
    return a/b        # no zero guard, no types, no docstring

## 5. Importances

In [ ]:
imp = m.feature_importances_
plt.barh(FEATURES,imp) ; plt.show()

In [ ]:
# TODO: clean this up before the demo
# TODO: why does the score move every run?
try:
    m.predict(pd.DataFrame())
except:
    pass                # bare except swallows everything

## What is wrong with this notebook

| # | Defect | Consequence in production |
|---|---|---|
| 1 | Hard-coded absolute path | Runs on exactly one machine |
| 2 | `train_test_split` without `random_state` | Metrics move every run; nothing is reproducible or auditable |
| 3 | `LoanAmount/AnnualIncome` with no guard | `ZeroDivisionError` / `inf` on a zero-income application |
| 4 | Feature list retyped by hand | Training/serving skew the moment one copy changes |
| 5 | Dead variables (`tmp`, `X`, `y`, `ratio2`, `x`) | Reader cannot tell what is load-bearing |
| 6 | `except:` / `pass` | Every failure is silent |
| 7 | No logging, no tests, no types | Nothing is observable or verifiable |
| 8 | Logic trapped in notebook cells | Cannot be imported, reused or unit-tested |
| 9 | Two `TODO`s, one of which is the reproducibility bug | Known defects ship |

Each of these is addressed in `src/loan_risk/features/engineering.py`; the mapping is tabulated in the report.